In [82]:
import numpy as np
import cv2
import os
import scipy.io as sio
from skimage import measure, morphology, filters, segmentation
from skimage.morphology import skeletonize, remove_small_objects
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt

In [ ]:
def km2deg(km):
    return km / 111.32  # Konversi km ke derajat

def calculate_area_km2(region_mask, resolution_km=1.0):
    pixel_area = resolution_km ** 2  
    return np.sum(region_mask) * pixel_area

In [ ]:
def fuzzy_skeleton_pruning(skeleton, max_iter=5, min_branch_length_km=3.0, resolution_km=1.0):
    min_branch_length_px = min_branch_length_km / resolution_km
    pruned = skeleton.copy()
    
    for _ in range(max_iter):
        #Temukan endpoints dan branch points
        endpoints = find_endpoints(pruned)
        branch_points = find_branchpoints(pruned)
        
        # Hitung distance transform dari branch points
        dist_map = distance_transform_edt(~branch_points)
        
        # Fuzzy logic: Hapus endpoint yang jaraknya > min_branch_length_px
        to_remove = endpoints & (dist_map > min_branch_length_px)
        pruned[to_remove] = 0
        
        # Re-skeletonize untuk mempertahankan konektivitas
        pruned = skeletonize(pruned)
    
    return pruned

def find_endpoints(skel):
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]])
    conv = cv2.filter2D(skel.astype(np.uint8), -1, kernel)
    return (conv == 11) & skel

def find_branchpoints(skel):
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]])
    conv = cv2.filter2D(skel.astype(np.uint8), -1, kernel)
    return (conv >= 13) & skel

In [ ]:
def calculate_principal_axis(skeleton, resolution_km=1.0):
    props = measure.regionprops(skeleton.astype(int))
    if not props:
        return None, None
    
    # Konversi ke km untuk perhitungan yang akurat
    coords = np.argwhere(props[0].image)
    y_km = coords[:,0] * resolution_km
    x_km = coords[:,1] * resolution_km
    
    # Hitung matriks kovarians
    cov = np.cov(x_km - np.mean(x_km), 
                y_km - np.mean(y_km))
    
    # Eigenvectors
    evals, evecs = np.linalg.eig(cov)
    orientation = np.arctan2(evecs[1,0], evecs[0,0])
    angle_deg = np.degrees(orientation) % 180
    
    # Eccentricity
    eccentricity = np.sqrt(1 - (min(evals) / max(evals)))
    
    return angle_deg, eccentricity

# Konfigurasi
data_path = 'D:/College/MENEA/FIXED_DATA/RUN_TEST/20220815'
output_path = 'D:/College/MENEA/FIXED_DATA/OUTPUT/20220815/skeletons/'
min_area_km2 = 10  # Minimum area 10 km2
resolution_km = 0.12  # Resolusi data radar dalam km/pixel

# Proses semua file .mat
for filename in [f for f in os.listdir(data_path) if f.endswith('.mat')]:
    try:
        # Load data radar
        data = sio.loadmat(os.path.join(data_path, filename))
        ZI = data['ZI'].astype(np.float32) / 1e4
        ZI[ZI < 0] = 0
        
        # Thresholding adaptif
        threshold = filters.threshold_otsu(ZI[ZI > 0])
        binary = (ZI > threshold).astype(np.uint8)
        
        # Hilangkan noise kecil
        cleaned = remove_small_objects(binary.astype(bool), min_size=50)
        cleaned = segmentation.clear_border(cleaned)
        
        # Label region
        label_img = measure.label(cleaned)
        regions = measure.regionprops(label_img, intensity_image=ZI)
        
        # Pilih region dengan area > 10 km2
        valid_regions = []
        for region in regions:
            area = calculate_area_km2(region.image, resolution_km)
            if area >= min_area_km2:
                valid_regions.append((region, area))
        
        if not valid_regions:
            print(f"Tidak ada region > {min_area_km2} km2 di {filename}")
            continue
        
        # Ambil region terbesar
        region, area = max(valid_regions, key=lambda x: x[1])
        roi = region.image
        
        # Buat skeleton
        skeleton = skeletonize(roi)
        
        # Prune skeleton
        pruned_skeleton = fuzzy_skeleton_pruning(skeleton, 
                                               min_branch_length_km=3.0,
                                               resolution_km=resolution_km)
        
        # Hitung properti
        angle, eccentricity = calculate_principal_axis(pruned_skeleton, resolution_km)
        
        # Visualisasi
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle(f"Principal Axis Analysis: {filename[:-4]}\n"
                    f"Area: {area:.1f} km² | Angle: {angle:.1f}° | Eccentricity: {eccentricity:.2f}", 
                    fontsize=12)
        
        # Original
        axes[0,0].imshow(ZI, cmap='jet')
        axes[0,0].set_title("Original Data")
        axes[0,0].axis('off')
        
        # Binary ROI
        axes[0,1].imshow(roi, cmap='gray')
        axes[0,1].set_title(f"Region of Interest\n{area:.1f} km²")
        axes[0,1].axis('off')
        
        # Original Skeleton
        axes[1,0].imshow(skeleton, cmap='gray')
        axes[1,0].set_title("Original Skeleton")
        axes[1,0].axis('off')
        
        # Pruned Skeleton with Axis
        axes[1,1].imshow(pruned_skeleton, cmap='gray')
        
        # Gambar principal axis
        if angle is not None:
            y, x = np.mean(np.where(pruned_skeleton), axis=1)
            length = max(pruned_skeleton.shape) * 0.4
            dx = length * np.cos(np.radians(angle))
            dy = length * np.sin(np.radians(angle))
            axes[1,1].plot([x - dx, x + dx], [y - dy, y + dy], 'r-', linewidth=2)
        
        axes[1,1].set_title("Pruned Skeleton with Principal Axis")
        axes[1,1].axis('off')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f"Principal_Axis_{filename[:-4]}.png"), 
                   bbox_inches='tight', dpi=300)
        plt.close()
        
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")

print("Processing complete!")

Processing complete!


In [11]:
import numpy as np
import cv2
import os
import scipy.io as sio
from skimage import measure, morphology, filters, segmentation, color
from skimage.morphology import skeletonize, remove_small_objects
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from matplotlib.colors import LinearSegmentedColormap
from datetime import datetime

# Date configuration
DATE_STR = "20221109"  # Format: YYYYMMDD
date_obj = datetime.strptime(DATE_STR, "%Y%m%d")
FORMATTED_DATE = date_obj.strftime("%d %B %Y")  # e.g., "04 February 2021"

def km2deg(km):
    return km / 111.32  # Konversi km ke derajat

def calculate_area_km2(region_mask, resolution_km=1.0):
    pixel_area = resolution_km ** 2  
    return np.sum(region_mask) * pixel_area

def fuzzy_skeleton_pruning(skeleton, max_iter=5, min_branch_length_km=3.0, resolution_km=1.0):
    min_branch_length_px = min_branch_length_km / resolution_km
    pruned = skeleton.copy()
    
    for _ in range(max_iter):
        # Temukan endpoints dan branch points
        endpoints = find_endpoints(pruned)
        branch_points = find_branchpoints(pruned)
        
        # Hitung distance transform dari branch points
        dist_map = distance_transform_edt(~branch_points)
        
        # Fuzzy logic: Hapus endpoint yang jaraknya > min_branch_length_px
        to_remove = endpoints & (dist_map > min_branch_length_px)
        pruned[to_remove] = 0
        
        # Re-skeletonize untuk mempertahankan konektivitas
        pruned = skeletonize(pruned)
    
    return pruned

def find_endpoints(skel):
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]])
    conv = cv2.filter2D(skel.astype(np.uint8), -1, kernel)
    return (conv == 11) & skel

def find_branchpoints(skel):
    kernel = np.array([[1, 1, 1],
                       [1, 10, 1],
                       [1, 1, 1]])
    conv = cv2.filter2D(skel.astype(np.uint8), -1, kernel)
    return (conv >= 13) & skel

def calculate_principal_axis(skeleton, resolution_km=1.0):
    props = measure.regionprops(skeleton.astype(int))
    if not props:
        return None, None
    
    # Konversi ke km untuk perhitungan yang akurat
    coords = np.argwhere(props[0].image)
    y_km = coords[:,0] * resolution_km
    x_km = coords[:,1] * resolution_km
    
    # Hitung matriks kovarians
    cov = np.cov(x_km - np.mean(x_km), 
                y_km - np.mean(y_km))
    
    # Eigenvectors
    evals, evecs = np.linalg.eig(cov)
    orientation = np.arctan2(evecs[1,0], evecs[0,0])
    angle_deg = np.degrees(orientation) % 180
    
    # Eccentricity
    eccentricity = np.sqrt(1 - (min(evals) / max(evals)))
    
    return angle_deg, eccentricity

# Konfigurasi
data_path = f'D:/College/MENEA/FIXED_DATA/RUN_TEST/{DATE_STR}'
output_path = f'D:/College/MENEA/FIXED_DATA/OUTPUT/{DATE_STR}/skeletons/'
min_area_km2 = 10  # Minimum area 10 km2
resolution_km = 0.12  # Resolusi data radar dalam km/pixel

# Create scientific colormaps
def create_rainbow_cmap():
    colors = [(0,0,0.5), (0,0,1), (0,1,1), (0,1,0), (1,1,0), (1,0,0), (0.5,0,0)]
    return LinearSegmentedColormap.from_list('sci_rainbow', colors)

sci_rainbow = create_rainbow_cmap()
sci_gray = LinearSegmentedColormap.from_list('sci_gray', [(0.9,0.9,0.9), (0.1,0.1,0.1)])

# Set style untuk plot
plt.style.use('default')
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'figure.dpi': 300
})

# Create output directory if not exists
os.makedirs(output_path, exist_ok=True)

# Proses semua file .mat
for filename in [f for f in os.listdir(data_path) if f.endswith('.mat')]:
    try:
        # Load data radar
        data = sio.loadmat(os.path.join(data_path, filename))
        ZI = data['ZI'].astype(np.float32) / 1e4
        ZI[ZI < 0] = 0
        
        # Thresholding adaptif
        threshold = filters.threshold_otsu(ZI[ZI > 0])
        binary = (ZI > threshold).astype(np.uint8)
        
        # Hilangkan noise kecil
        cleaned = remove_small_objects(binary.astype(bool), min_size=50)
        cleaned = segmentation.clear_border(cleaned)
        
        # Label region
        label_img = measure.label(cleaned)
        regions = measure.regionprops(label_img, intensity_image=ZI)
        
        # Pilih region dengan area > 10 km2
        valid_regions = []
        for region in regions:
            area = calculate_area_km2(region.image, resolution_km)
            if area >= min_area_km2:
                valid_regions.append((region, area))
        
        if not valid_regions:
            print(f"Tidak ada region > {min_area_km2} km2 di {filename}")
            continue
        
        # Ambil region terbesar
        region, area = max(valid_regions, key=lambda x: x[1])
        roi = region.image
        
        # Buat skeleton (tetap dihitung tapi tidak diplot)
        skeleton = skeletonize(roi)
        pruned_skeleton = fuzzy_skeleton_pruning(skeleton, 
                                               min_branch_length_km=3.0,
                                               resolution_km=resolution_km)
        angle, eccentricity = calculate_principal_axis(pruned_skeleton, resolution_km)
        
        # Visualisasi dengan tampilan paper ilmiah
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))  # 1 row, 3 columns
        
        # Atur judul utama
        fig.suptitle(f"Bow Echo Analysis - {FORMATTED_DATE}\nArea: {area:.1f} km² | Angle: {angle:.1f}° | Eccentricity: {eccentricity:.2f}", 
                    fontsize=12, y=1.02)
        
        # Original Data dengan colormap rainbow ilmiah (tanpa colorbar)
        axs[0].imshow(ZI, cmap=sci_rainbow, vmin=0, vmax=50)
        axs[0].set_title("Reflectivity (dBZ)", fontsize=11)
        axs[0].axis('off')
        
        # Binary ROI dengan colormap grayscale ilmiah (tanpa colorbar)
        axs[1].imshow(roi, cmap=sci_gray)
        axs[1].set_title(f"Bow Echo Region\n{area:.1f} km²", fontsize=11)
        axs[1].axis('off')
        
        # Original Skeleton dengan colormap merah (tanpa colorbar)
        axs[2].imshow(skeleton, cmap='Reds')
        axs[2].set_title("Bow Echo Skeleton", fontsize=11)
        axs[2].axis('off')
        
        # Atur layout
        plt.tight_layout(pad=2.0)
        plt.savefig(os.path.join(output_path, f"Storm_Analysis_{filename[:-4]}.png"), 
                   bbox_inches='tight', dpi=300)
        plt.close()
        
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")

print("Processing complete!")

Processing complete!
